In [1]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working directory:", Path.cwd())

Python: c:\Users\megdo\Desktop\underwater-object-detection\venv\Scripts\python.exe
Working directory: c:\Users\megdo\Desktop\underwater-object-detection\notebooks


In [2]:
from pathlib import Path
import json

# ============================================================
# PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().parent

# Fallback in case notebook is run from the project root
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path.cwd()

print("Project root:", PROJECT_ROOT)


# ============================================================
# ENHANCED DUO DATASET PATHS
# ============================================================

# Enhanced images created using the image-enhancement script
IMAGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "DUO_enhanced"
    / "images"
)

TRAIN_IMAGE_DIR = IMAGE_DIR / "train"
TEST_IMAGE_DIR = IMAGE_DIR / "test"


# Enhanced dataset annotations
# These are copies of the original COCO annotations because
# image enhancement does not change bounding-box coordinates.
ANNOTATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "DUO_enhanced"
    / "annotations"
)


# ============================================================
# COCO ANNOTATION FILES
# ============================================================

clean_train_path = (
    ANNOTATION_DIR
    / "instances_train_clean.json"
)

clean_test_path = (
    ANNOTATION_DIR
    / "instances_test_clean.json"
)

train_split_path = (
    ANNOTATION_DIR
    / "instances_train_split.json"
)

val_split_path = (
    ANNOTATION_DIR
    / "instances_val_split.json"
)


# ============================================================
# EXISTING SPLIT DIRECTORY
# ============================================================

SPLIT_DIR = (
    PROJECT_ROOT
    / "data"
    / "splits"
)


# ============================================================
# NEW OUTPUT DIRECTORY
# ============================================================

# IMPORTANT:
# This is a different folder from the original baseline so the
# original Faster R-CNN results are not overwritten.

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "faster_rcnn"
    / "duo_faster_rcnn_enhanced"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CHECK PATHS
# ============================================================

print("\nEnhanced DUO paths:")
print("Train images:", TRAIN_IMAGE_DIR)
print("Test images:", TEST_IMAGE_DIR)
print("Annotations:", ANNOTATION_DIR)
print("Output directory:", OUTPUT_DIR)

print("\nPath checks:")
print("Train images exist:", TRAIN_IMAGE_DIR.exists())
print("Test images exist:", TEST_IMAGE_DIR.exists())
print("Train split exists:", train_split_path.exists())
print("Validation split exists:", val_split_path.exists())
print("Test annotations exist:", clean_test_path.exists())

Project root: c:\Users\megdo\Desktop\underwater-object-detection

Enhanced DUO paths:
Train images: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\images\train
Test images: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\images\test
Annotations: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\annotations
Output directory: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_enhanced

Path checks:
Train images exist: True
Test images exist: True
Train split exists: True
Validation split exists: True
Test annotations exist: True


In [3]:
with clean_train_path.open("r", encoding="utf-8") as file: # open the clean training JSON file in read mode with UTF-8 encoding
    clean_train_data = json.load(file)

with clean_test_path.open("r", encoding="utf-8") as file: # open the clean test JSON file in read mode with UTF-8 encoding
    clean_test_data = json.load(file)

train_filenames = { #   create a set of training filenames by reading the train.txt file in the splits directory, stripping whitespace from each line, and including only non-empty lines
    line.strip()
    for line in (SPLIT_DIR / "train.txt").read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
}

val_filenames = { #   create a set of validation filenames by reading the val.txt file in the splits directory, stripping whitespace from each line, and including only non-empty lines
    line.strip()
    for line in (SPLIT_DIR / "val.txt").read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
}

print("Training filenames:", len(train_filenames))
print("Validation filenames:", len(val_filenames))
print("Official test images:", len(clean_test_data["images"]))

Training filenames: 5336
Validation filenames: 1335
Official test images: 1111


In [4]:
def create_coco_subset(coco_data, selected_filenames): # define a function to create a COCO subset from the given COCO data and selected filenames
    selected_images = [
        image
        for image in coco_data["images"]
        if image["file_name"] in selected_filenames
    ]

    selected_image_ids = { # create a set of selected image IDs from the selected images
        image["id"]
        for image in selected_images
    }

    selected_annotations = [ #  create a list of selected annotations from the COCO data, including only those annotations whose image ID is in the set of selected image IDs
        annotation
        for annotation in coco_data["annotations"]
        if annotation["image_id"] in selected_image_ids
    ]

    subset = { # create a dictionary representing the COCO subset, including the selected images, selected annotations, and categories from the original COCO data
        "images": selected_images,
        "annotations": selected_annotations,
        "categories": coco_data["categories"],
    }

    if "info" in coco_data: # check if the info key exists in the original COCO data and, if so, include it in the subset
        subset["info"] = coco_data["info"]

    if "licenses" in coco_data: # check if the licenses key exists in the original COCO data and, if so, include it in the subset
        subset["licenses"] = coco_data["licenses"]

    return subset # return the created COCO subset


train_split_data = create_coco_subset( # create the training split data by calling the create_coco_subset function with the clean training data and training filenames
    clean_train_data,
    train_filenames,
)

val_split_data = create_coco_subset( # create the validation split data by calling the create_coco_subset function with the clean training data and validation filenames

    clean_train_data,
    val_filenames,
)

print(
    "Training split:",
    len(train_split_data["images"]),
    "images and",
    len(train_split_data["annotations"]),
    "annotations",
)

print(
    "Validation split:",
    len(val_split_data["images"]),
    "images and",
    len(val_split_data["annotations"]),
    "annotations",
)

Training split: 5336 images and 51500 annotations
Validation split: 1335 images and 12497 annotations


In [5]:
train_split_path = ( # set the training split JSON path to the instances_train_split.json file within the
    ANNOTATION_DIR
    / "instances_train_split.json"
)

val_split_path = ( # set the validation split JSON path to the instances_val_split.json file within the
    ANNOTATION_DIR
    / "instances_val_split.json"
)

with train_split_path.open("w", encoding="utf-8") as file: # open the training split JSON file in write mode with UTF-8 encoding
    json.dump(train_split_data, file)

with val_split_path.open("w", encoding="utf-8") as file: # open the validation split JSON file in write mode with UTF-8 encoding
    json.dump(val_split_data, file)

print("Saved:", train_split_path)
print("Saved:", val_split_path)

Saved: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\annotations\instances_train_split.json
Saved: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\annotations\instances_val_split.json


In [6]:
#check there is no overlap between train and val splits
train_names_check = {
    image["file_name"]
    for image in train_split_data["images"]
}

val_names_check = { # create a set of validation image filenames by extracting the "file_name" from each image in the validation split data
    image["file_name"]
    for image in val_split_data["images"]
}

overlap = train_names_check & val_names_check # compute the intersection of the training and validation image filename sets to find any overlapping filenames

print("Training images:", len(train_names_check))
print("Validation images:", len(val_names_check))
print("Overlapping filenames:", len(overlap))

Training images: 5336
Validation images: 1335
Overlapping filenames: 0


In [7]:
import sys # import the sys module to access system-specific parameters and functions
import torch # import the torch module for PyTorch functionalities
import torchvision # import the torchvision module for computer vision functionalities in PyTorch

print("Python:", sys.executable)
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: c:\Users\megdo\Desktop\underwater-object-detection\venv\Scripts\python.exe
PyTorch version: 2.13.0+cpu
Torchvision version: 0.28.0+cpu
CUDA available: False


In [8]:
TRAIN_IMAGE_DIR = PROJECT_ROOT / "data" / "processed" / "DUO_enhanced" / "images" / "train"
TEST_IMAGE_DIR = PROJECT_ROOT / "data" / "processed" / "DUO_enhanced" / "images" / "test"

TRAIN_JSON = ANNOTATION_DIR / "instances_train_split.json" # set the training JSON path to the instances_train_split.json file within the annotation directory
VAL_JSON = ANNOTATION_DIR / "instances_val_split.json" # set the validation JSON path to the instances_val_split.json file within the annotation directory
TEST_JSON = ANNOTATION_DIR / "instances_test_clean.json" # set the test JSON path to the instances_test_clean.json file within the annotation directory

print("Training images found:", TRAIN_IMAGE_DIR.exists())
print("Test images found:", TEST_IMAGE_DIR.exists())
print("Training JSON found:", TRAIN_JSON.exists())
print("Validation JSON found:", VAL_JSON.exists())
print("Test JSON found:", TEST_JSON.exists())

Training images found: True
Test images found: True
Training JSON found: True
Validation JSON found: True
Test JSON found: True


In [9]:
import sys
import json
import csv
import time
from pathlib import Path

import torch
import torchvision
import numpy as np
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F

from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: c:\Users\megdo\Desktop\underwater-object-detection\venv\Scripts\python.exe
PyTorch: 2.13.0+cpu
Torchvision: 0.28.0+cpu
CUDA available: False


In [10]:
PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path.cwd()

# Enhanced DUO images
TRAIN_IMAGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "DUO_enhanced"
    / "images"
    / "train"
)

TEST_IMAGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "DUO_enhanced"
    / "images"
    / "test"
)

# Enhanced annotation directory
ANNOTATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "DUO_enhanced"
    / "annotations"
)

TRAIN_JSON = ANNOTATION_DIR / "instances_train_split.json"
VAL_JSON = ANNOTATION_DIR / "instances_val_split.json"
TEST_JSON = ANNOTATION_DIR / "instances_test_clean.json"

# Separate results folder
RUN_DIR = (
    PROJECT_ROOT
    / "results"
    / "faster_rcnn"
    / "duo_faster_rcnn_enhanced"
)

RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Train images:", TRAIN_IMAGE_DIR)
print("Test images:", TEST_IMAGE_DIR)
print("Training JSON:", TRAIN_JSON)
print("Validation JSON:", VAL_JSON)
print("Test JSON:", TEST_JSON)
print("Results folder:", RUN_DIR)

print("\nPath checks:")
print("Train images:", TRAIN_IMAGE_DIR.exists())
print("Test images:", TEST_IMAGE_DIR.exists())
print("Train JSON:", TRAIN_JSON.exists())
print("Validation JSON:", VAL_JSON.exists())
print("Test JSON:", TEST_JSON.exists())

Project root: c:\Users\megdo\Desktop\underwater-object-detection
Train images: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\images\train
Test images: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\images\test
Training JSON: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\annotations\instances_train_split.json
Validation JSON: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\annotations\instances_val_split.json
Test JSON: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\annotations\instances_test_clean.json
Results folder: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_enhanced

Path checks:
Train images: True
Test images: True
Train JSON: True
Validation JSON: True
Test JSON: True


In [11]:
class DUOCocoDataset(Dataset):
    def __init__(self, image_dir, annotation_file):
        self.image_dir = Path(image_dir)

        with open(annotation_file, "r", encoding="utf-8") as f:
            self.coco = json.load(f)

        self.images = self.coco["images"]
        self.annotations = self.coco["annotations"]
        self.categories = self.coco["categories"]

        # Map annotations to image IDs
        self.annotations_by_image = {}

        for ann in self.annotations:
            image_id = ann["image_id"]

            if image_id not in self.annotations_by_image:
                self.annotations_by_image[image_id] = []

            self.annotations_by_image[image_id].append(ann)

        # Map original COCO category IDs to consecutive Faster R-CNN labels
        # 0 is reserved for background.
        category_ids = sorted(
            [category["id"] for category in self.categories]
        )

        self.category_to_label = {
            category_id: index + 1
            for index, category_id in enumerate(category_ids)
        }

        self.label_to_category = {
            value: key
            for key, value in self.category_to_label.items()
        }

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_info = self.images[index]

        image_id = image_info["id"]
        file_name = image_info["file_name"]

        image_path = self.image_dir / file_name

        image = Image.open(image_path).convert("RGB")
        image = F.to_tensor(image)

        anns = self.annotations_by_image.get(image_id, [])

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, width, height = ann["bbox"]

            # Ignore invalid boxes
            if width <= 0 or height <= 0:
                continue

            boxes.append([
                x,
                y,
                x + width,
                y + height
            ])

            labels.append(
                self.category_to_label[ann["category_id"]]
            )

            areas.append(
                ann.get("area", width * height)
            )

            iscrowd.append(
                ann.get("iscrowd", 0)
            )

        if len(boxes) > 0:
            boxes = torch.tensor(
                boxes,
                dtype=torch.float32
            )

            labels = torch.tensor(
                labels,
                dtype=torch.int64
            )

            areas = torch.tensor(
                areas,
                dtype=torch.float32
            )

            iscrowd = torch.tensor(
                iscrowd,
                dtype=torch.int64
            )

        else:
            boxes = torch.zeros(
                (0, 4),
                dtype=torch.float32
            )

            labels = torch.zeros(
                (0,),
                dtype=torch.int64
            )

            areas = torch.zeros(
                (0,),
                dtype=torch.float32
            )

            iscrowd = torch.zeros(
                (0,),
                dtype=torch.int64
            )

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor(
                image_id,
                dtype=torch.int64
            ),
            "area": areas,
            "iscrowd": iscrowd
        }

        return image, target

In [12]:
train_dataset = DUOCocoDataset(
    TRAIN_IMAGE_DIR,
    TRAIN_JSON
)

val_dataset = DUOCocoDataset(
    TRAIN_IMAGE_DIR,
    VAL_JSON
)

test_dataset = DUOCocoDataset(
    TEST_IMAGE_DIR,
    TEST_JSON
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

Training images: 5336
Validation images: 1335
Test images: 1111


In [13]:
image, target = train_dataset[0]

print("Image shape:", image.shape)
print("Boxes:", target["boxes"].shape)
print("Labels:", target["labels"].shape)
print("Image ID:", target["image_id"])
print("Pixel range:", float(image.min()), "to", float(image.max()))

Image shape: torch.Size([3, 405, 720])
Boxes: torch.Size([2, 4])
Labels: torch.Size([2])
Image ID: tensor(1)
Pixel range: 0.0 to 1.0


In [14]:
def collate_fn(batch):
    return tuple(zip(*batch))


BATCH_SIZE = 1

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 5336
Validation batches: 1335
Test batches: 1111


In [15]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


In [16]:
NUM_CLASSES = 5
# background + 4 DUO classes

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights,
    min_size=416,
    max_size=416
)

# Replace COCO classifier with DUO classifier
in_features = (
    model.roi_heads.box_predictor.cls_score.in_features
)

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

model.to(device)

print("Faster R-CNN ResNet50-FPN created.")
print("Number of classes:", NUM_CLASSES)

Faster R-CNN ResNet50-FPN created.
Number of classes: 5


In [17]:
params = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

print("Optimizer:", optimizer.__class__.__name__)
print("Initial learning rate:", optimizer.param_groups[0]["lr"])
print("Scheduler step size: 3")
print("Scheduler gamma: 0.1")

Optimizer: SGD
Initial learning rate: 0.005
Scheduler step size: 3
Scheduler gamma: 0.1


In [18]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    batches = 0

    for images, targets in loader:
        images = [
            image.to(device)
            for image in images
        ]

        targets = [
            {
                key: value.to(device)
                if torch.is_tensor(value)
                else value
                for key, value in target.items()
            }
            for target in targets
        ]

        loss_dict = model(
            images,
            targets
        )

        losses = sum(
            loss
            for loss in loss_dict.values()
        )

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        batches += 1

    return total_loss / batches

In [19]:
def calculate_validation_loss(model, loader, device):
    # Detection models return losses while in train mode.
    # Gradients are disabled so parameters are not updated.
    model.train()

    total_loss = 0.0
    batches = 0

    with torch.no_grad():

        for images, targets in loader:

            images = [
                image.to(device)
                for image in images
            ]

            targets = [
                {
                    key: value.to(device)
                    if torch.is_tensor(value)
                    else value
                    for key, value in target.items()
                }
                for target in targets
            ]

            loss_dict = model(
                images,
                targets
            )

            losses = sum(
                loss
                for loss in loss_dict.values()
            )

            total_loss += losses.item()
            batches += 1

    return total_loss / batches

In [20]:
NUM_EPOCHS = 8

LATEST_CHECKPOINT = (
    RUN_DIR
    / "latest_checkpoint.pth"
)

HISTORY_CSV = (
    RUN_DIR
    / "training_history.csv"
)

print("Run directory:", RUN_DIR)
print("Latest checkpoint:", LATEST_CHECKPOINT)
print("History:", HISTORY_CSV)
print("Maximum epochs:", NUM_EPOCHS)

Run directory: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_enhanced
Latest checkpoint: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_enhanced\latest_checkpoint.pth
History: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_enhanced\training_history.csv
Maximum epochs: 8


In [21]:
history = []

best_val_loss = float("inf")
best_epoch = None

for epoch in range(1, NUM_EPOCHS + 1):

    print(
        f"\n{'=' * 60}\n"
        f"Epoch {epoch}/{NUM_EPOCHS}\n"
        f"{'=' * 60}"
    )

    start_time = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    val_loss = calculate_validation_loss(
        model,
        val_loader,
        device
    )

    epoch_time = time.time() - start_time

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Train loss: {train_loss:.6f}"
    )

    print(
        f"Validation loss: {val_loss:.6f}"
    )

    print(
        f"Learning rate: {current_lr:.8f}"
    )

    print(
        f"Epoch duration: "
        f"{epoch_time / 60:.2f} minutes"
    )

    # Save epoch checkpoint
    checkpoint_path = (
        RUN_DIR
        / f"checkpoint_epoch_{epoch:02d}.pth"
    )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
        },
        checkpoint_path
    )

    # Save latest checkpoint
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
        },
        LATEST_CHECKPOINT
    )

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "learning_rate": current_lr,
            "duration_seconds": epoch_time
        }
    )

    pd.DataFrame(history).to_csv(
        HISTORY_CSV,
        index=False
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch

        best_path = RUN_DIR / "best_checkpoint.pth"

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
            },
            best_path
        )

        print(
            f"New best checkpoint: epoch {epoch}"
        )

    scheduler.step()

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)


Epoch 1/8
Train loss: 0.639333
Validation loss: 0.552694
Learning rate: 0.00500000
Epoch duration: 137.50 minutes
New best checkpoint: epoch 1

Epoch 2/8
Train loss: 0.502608
Validation loss: 0.480900
Learning rate: 0.00500000
Epoch duration: 134.21 minutes
New best checkpoint: epoch 2

Epoch 3/8
Train loss: 0.470914
Validation loss: 0.458742
Learning rate: 0.00500000
Epoch duration: 178.11 minutes
New best checkpoint: epoch 3

Epoch 4/8
Train loss: 0.348289
Validation loss: 0.388908
Learning rate: 0.00050000
Epoch duration: 132.80 minutes
New best checkpoint: epoch 4

Epoch 5/8
Train loss: 0.324802
Validation loss: 0.387655
Learning rate: 0.00050000
Epoch duration: 134.64 minutes
New best checkpoint: epoch 5

Epoch 6/8
Train loss: 0.311647
Validation loss: 0.385252
Learning rate: 0.00050000
Epoch duration: 130.61 minutes
New best checkpoint: epoch 6

Epoch 7/8
Train loss: 0.292869
Validation loss: 0.391205
Learning rate: 0.00005000
Epoch duration: 130.47 minutes

Epoch 8/8
Train loss

In [22]:
BEST_CHECKPOINT = (
    PROJECT_ROOT
    / "results"
    / "faster_rcnn"
    / "duo_faster_rcnn_enhanced"
    / "best_checkpoint.pth"
)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("Checkpoint loaded:", BEST_CHECKPOINT)
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])
print("Test image directory:", TEST_IMAGE_DIR)

Checkpoint loaded: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_enhanced\best_checkpoint.pth
Best epoch: 6
Validation loss: 0.38525154962926433
Test image directory: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_enhanced\images\test


In [23]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import time

metric = MeanAveragePrecision(
    box_format="xyxy",
    iou_type="bbox",
    class_metrics=True
)

model.eval()

total_inference_time = 0.0
total_images = 0

print("Starting enhanced DUO test evaluation...")

with torch.no_grad():
    for batch_number, (images, targets) in enumerate(test_loader, start=1):

        images = [image.to(device) for image in images]

        start_time = time.perf_counter()

        predictions = model(images)

        end_time = time.perf_counter()

        total_inference_time += end_time - start_time
        total_images += len(images)

        predictions_cpu = [
            {
                "boxes": prediction["boxes"].cpu(),
                "scores": prediction["scores"].cpu(),
                "labels": prediction["labels"].cpu()
            }
            for prediction in predictions
        ]

        targets_cpu = [
            {
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu()
            }
            for target in targets
        ]

        metric.update(
            predictions_cpu,
            targets_cpu
        )

        if batch_number % 100 == 0:
            print(
                f"Evaluated {batch_number}/{len(test_loader)} batches"
            )

results = metric.compute()

map_50_95 = results["map"].item()
map_50 = results["map_50"].item()

average_inference_ms = (
    total_inference_time / total_images
) * 1000

fps = total_images / total_inference_time

per_class_ap = results["map_per_class"]
class_labels = results["classes"]

print("\nEnhanced DUO test evaluation completed.")
print(f"mAP@0.5:0.95: {map_50_95:.6f}")
print(f"mAP@0.5: {map_50:.6f}")
print(f"Average inference time: {average_inference_ms:.2f} ms/image")
print(f"Approximate FPS: {fps:.2f}")
print("Per-class AP:", per_class_ap)
print("Class labels:", class_labels)

c:\Users\megdo\Desktop\underwater-object-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting enhanced DUO test evaluation...
Evaluated 100/1111 batches
Evaluated 200/1111 batches
Evaluated 300/1111 batches
Evaluated 400/1111 batches
Evaluated 500/1111 batches
Evaluated 600/1111 batches
Evaluated 700/1111 batches
Evaluated 800/1111 batches
Evaluated 900/1111 batches
Evaluated 1000/1111 batches
Evaluated 1100/1111 batches

Enhanced DUO test evaluation completed.
mAP@0.5:0.95: 0.459580
mAP@0.5: 0.691373
Average inference time: 1008.50 ms/image
Approximate FPS: 0.99
Per-class AP: tensor([0.4535, 0.6055, 0.2649, 0.5144])
Class labels: tensor([1, 2, 3, 4], dtype=torch.int32)
